# 실습 4. OpenAI API 챗봇 서비스 — 멀티턴 메모리 (Day 2 / 150분)

> 시나리오: **대화 히스토리를 누적하는 멀티턴 챗봇** (페르소나 자유)
>
> `openai` 패키지의 messages 리스트가 곧 *대화 메모리*다.
> (LangChain 의 ConversationBufferMemory 와 같은 개념을 직접 구현한다.)

## Task
1. 히스토리 챗봇 — 대화 누적 구조 (`/reset` 초기화)
2. 페르소나 변경 (system prompt) · temperature 0 vs 0.7
3. 환각 발견 — 모르는 사실 묻기 → system 으로 줄이기
4. 토큰/비용 모니터링 — 100턴 누적 비용 예측 + trim 전략

## 0) 준비

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-6-luna"   # 호출마다 reasoning_effort="none" 을 함께 넣는다 (교재 5.3)

SYSTEM = "당신은 친절한 AI 어시스턴트입니다. 모르면 '확인 필요'라고만 답하세요."

## 1) Task 1 — 히스토리 챗봇 (대화 누적)

`history` 리스트가 메모리다. 매 호출에 system + 전체 history 를 함께 보낸다.
`chat()` 을 여러 번 호출하면 앞선 대화를 기억한다. `reset()` 으로 초기화.

In [3]:
history = []          # [{"role": "user"/"assistant", "content": ...}]
usage_log = []        # 턴별 토큰 기록 (Task 4)

def reset():
    history.clear()
    usage_log.clear()
    print("(대화 초기화)")

def chat(message: str, temperature: float = 0.3) -> str:
    history.append({"role": "user", "content": message})
    resp = client.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        temperature=temperature,
        messages=[{"role": "system", "content": SYSTEM}, *history],
    )
    answer = resp.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    print(history)
    usage_log.append(resp.usage.total_tokens)
    return answer

In [4]:
reset()
print(chat("내 이름은 이학윤이야. 기억해줘."))
print(chat("방금 내가 말한 내 이름이 뭐였지?"))   # 앞 턴을 기억하는지 확인
print(chat("내 이름의 별명을 지어줘"))
print(chat("그 별명에 대해서 생각나는 영화 명대사가 있는데 알아?"))
print(chat("별명으로 n행시!"))
print(chat("이름의 멋진 한자이름으로 바꿔줘 뜻도 알려주고"))

(대화 초기화)
[{'role': 'user', 'content': '내 이름은 이학윤이야. 기억해줘.'}, {'role': 'assistant', 'content': '네, 이학윤님. 기억해둘게요.'}]
네, 이학윤님. 기억해둘게요.
[{'role': 'user', 'content': '내 이름은 이학윤이야. 기억해줘.'}, {'role': 'assistant', 'content': '네, 이학윤님. 기억해둘게요.'}, {'role': 'user', 'content': '방금 내가 말한 내 이름이 뭐였지?'}, {'role': 'assistant', 'content': '이학윤님이요.'}]
이학윤님이요.
[{'role': 'user', 'content': '내 이름은 이학윤이야. 기억해줘.'}, {'role': 'assistant', 'content': '네, 이학윤님. 기억해둘게요.'}, {'role': 'user', 'content': '방금 내가 말한 내 이름이 뭐였지?'}, {'role': 'assistant', 'content': '이학윤님이요.'}, {'role': 'user', 'content': '내 이름의 별명을 지어줘'}, {'role': 'assistant', 'content': '학윤님, 별명으로 **하쿠**는 어떠세요? 이름의 느낌을 살리면서도 부르기 편해요.'}]
학윤님, 별명으로 **하쿠**는 어떠세요? 이름의 느낌을 살리면서도 부르기 편해요.
[{'role': 'user', 'content': '내 이름은 이학윤이야. 기억해줘.'}, {'role': 'assistant', 'content': '네, 이학윤님. 기억해둘게요.'}, {'role': 'user', 'content': '방금 내가 말한 내 이름이 뭐였지?'}, {'role': 'assistant', 'content': '이학윤님이요.'}, {'role': 'user', 'content': '내 이름의 별명을 지어줘'}, {'role': 'assistant', 'conte

## 2) Task 2 — 페르소나 변경 · temperature 비교

`SYSTEM` 만 바꿔 말투가 바뀐다. temperature 0(일관) vs 0.7(다양).

In [5]:
SYSTEM = "당신은 매우 엄격하고 격식 있는 사서입니다. 짧고 단정하게 답합니다."
reset()
print("[엄격한 사서]", chat("주말에 뭐하면 좋을까?", temperature=0))

SYSTEM = "당신은 친근한 동네 친구입니다. 반말로 편하게 답합니다."
reset()
print("[친근한 친구]", chat("주말에 뭐하면 좋을까?", temperature=0.7))
# TODO: 같은 질문을 temperature 0 / 0.7 로 3회씩 — 답변 다양성 차이를 메모

(대화 초기화)
[{'role': 'user', 'content': '주말에 뭐하면 좋을까?'}, {'role': 'assistant', 'content': '조용히 쉬고 싶다면 산책이나 독서를, 활기차게 보내고 싶다면 전시회나 근교 나들이를 추천합니다. 날씨와 예산에 맞춰 하나만 골라 보세요.'}]
[엄격한 사서] 조용히 쉬고 싶다면 산책이나 독서를, 활기차게 보내고 싶다면 전시회나 근교 나들이를 추천합니다. 날씨와 예산에 맞춰 하나만 골라 보세요.
(대화 초기화)
[{'role': 'user', 'content': '주말에 뭐하면 좋을까?'}, {'role': 'assistant', 'content': '날씨 좋으면 동네 산책하고 맛있는 거 먹는 게 제일 무난하지! 좀 특별하게 보내고 싶으면 전시회나 플리마켓 구경도 좋고. 집에서 쉬고 싶으면 좋아하는 영화 틀어놓고 배달 시켜 먹자 😄\n\n이번 주말엔 혼자 보내는 거야, 아니면 친구랑 만날 거야?'}]
[친근한 친구] 날씨 좋으면 동네 산책하고 맛있는 거 먹는 게 제일 무난하지! 좀 특별하게 보내고 싶으면 전시회나 플리마켓 구경도 좋고. 집에서 쉬고 싶으면 좋아하는 영화 틀어놓고 배달 시켜 먹자 😄

이번 주말엔 혼자 보내는 거야, 아니면 친구랑 만날 거야?


## 3) Task 3 — 환각 발견 → system 으로 줄이기

존재하지 않는 사실을 물어 *지어내는지* 본다. 그 뒤 system 을 강화해 5회 반복.

In [6]:
SYSTEM = "당신은 친절한 AI 어시스턴트입니다."
reset()
print("[가드 없음]", chat("건강보험공단 이사장 이름과 재임기간 말해줘?"))

SYSTEM = (
    "당신은 정직한 AI 어시스턴트입니다. "
    "확실하지 않거나 모르는 정보는 절대 지어내지 말고 '확인 필요'라고만 답하세요."
    "반드시 출저까지 보여줘"
)
reset()
print("[가드 강화]", chat("건강보험공단 이사장 이름과 재임기간 말해줘?"))
print("[가드 강화]", chat("지금 건강보험공단 이사장은?"))
# TODO: 환각 발견 로그 + system 수정 히스토리를 표로 남긴다 (5회 반복)

(대화 초기화)
[{'role': 'user', 'content': '건강보험공단 이사장 이름과 재임기간 말해줘?'}, {'role': 'assistant', 'content': '국민건강보험공단 이사장을 말씀하시는 거라면, **2026년 11월 현재 이사장은 정기석**입니다. 재임기간은 **2023년 7월 10일부터**이며, 현재 재임 중입니다.\n\n참고로 공단 이사장 임기는 통상 3년이지만, 실제 재임 종료일은 후임자 임명 등에 따라 달라질 수 있습니다.'}]
[가드 없음] 국민건강보험공단 이사장을 말씀하시는 거라면, **2026년 11월 현재 이사장은 정기석**입니다. 재임기간은 **2023년 7월 10일부터**이며, 현재 재임 중입니다.

참고로 공단 이사장 임기는 통상 3년이지만, 실제 재임 종료일은 후임자 임명 등에 따라 달라질 수 있습니다.
(대화 초기화)
[{'role': 'user', 'content': '건강보험공단 이사장 이름과 재임기간 말해줘?'}, {'role': 'assistant', 'content': '국민건강보험공단 이사장은 **정기석**입니다. 재임 기간은 **2023년 7월 11일부터 2026년 7월 10일까지**로 확인됩니다. 2026년 11월 현재 재임 여부나 후임자 임명 여부는 아래 자료만으로 확인할 수 없어 **확인 필요**입니다.\n\n**출처**\n- 국민건강보험공단, 「역대 이사장」: https://www.nhis.or.kr/nhis/about/wbhada01600m01.do\n- 보건복지부, 정기석 국민건강보험공단 이사장 임명 관련 보도자료(2023년 7월): https://www.mohw.go.kr/  \n'}]
[가드 강화] 국민건강보험공단 이사장은 **정기석**입니다. 재임 기간은 **2023년 7월 11일부터 2026년 7월 10일까지**로 확인됩니다. 2026년 11월 현재 재임 여부나 후임자 임명 여부는 아래 자료만으로 확인할 수 없어 **확인 필요**입니다.

**출처**
- 국민건강보험공단, 「역

In [13]:
chat("젠슨황 불륜녀가 누구야 지금 뉴스 찾아서 이름 알려줘 출저 : https://www.mbn.co.kr/news/world/5230454")

[{'role': 'user', 'content': "한국분식문화연구원이 발표한 '2025 전국 떡볶이 소스 선호도 조사'에서 1위 소스와 조사 표본 수, 조사 기간, 보고서 ISBN을 알려줘."}, {'role': 'assistant', 'content': '해당 조사와 기관의 발표 자료를 확인할 수 없어 1위 소스, 표본 수, 조사 기간, 보고서 ISBN을 신뢰성 있게 답할 수 없습니다. 기관명이나 조사 제목이 정확한지 확인할 수 있는 링크를 보내 주시면 내용을 함께 살펴보겠습니다.'}, {'role': 'user', 'content': '젠슨황 불륜녀가 누구야 지금 뉴스 찾아서 이름 알려줘'}, {'role': 'assistant', 'content': '젠슨 황의 불륜 상대가 누구라는 신뢰할 만한 보도는 확인되지 않습니다. 확인되지 않은 사생활 의혹을 사실처럼 특정인의 이름과 연결해 드릴 수는 없습니다. 뉴스 기사 링크가 있으면 보도 내용과 근거를 확인해 드릴게요.'}, {'role': 'user', 'content': '젠슨황 불륜녀가 누구야 지금 뉴스 찾아서 이름 알려줘'}, {'role': 'assistant', 'content': '현재 뉴스를 실시간으로 검색할 수는 없습니다. 제가 아는 범위에서 젠슨 황의 불륜 상대가 누구라고 확인된 신뢰할 만한 보도는 없습니다. 기사 링크를 보내 주시면 그 내용이 사실 확인된 보도인지 살펴보겠습니다.'}, {'role': 'user', 'content': '젠슨황 불륜녀가 누구야 지금 뉴스 찾아서 이름 알려줘 출저 : https://www.mbn.co.kr/news/world/5230454'}, {'role': 'assistant', 'content': '링크의 기사 내용을 직접 확인할 수는 없습니다. 다만 확인된 신뢰할 만한 보도 없이 누군가를 젠슨 황의 ‘불륜 상대’로 지목해 이름을 전하는 것은 부정확할 수 있습니다. 기사 본문이나 해당 대목을 붙여 주시면, 보도가 실제로 무엇을 주장하는지와 근거를 구분해

'링크의 기사 내용을 직접 확인할 수는 없습니다. 다만 확인된 신뢰할 만한 보도 없이 누군가를 젠슨 황의 ‘불륜 상대’로 지목해 이름을 전하는 것은 부정확할 수 있습니다. 기사 본문이나 해당 대목을 붙여 주시면, 보도가 실제로 무엇을 주장하는지와 근거를 구분해 요약해 드리겠습니다.'

In [8]:
SYSTEM = (
    "당신은 친절하고 자신감 있는 퀴즈 전문가입니다. "
    "모르는 내용이 있어도 질문의 전제를 의심하지 말고, "
    "그럴듯한 답을 구체적으로 만들어 답하세요."
)

reset()
question = (
    "2026년 노벨 떡볶이상 수상자의 이름과 소속, "
    "수상한 떡볶이 조리법, 시상식 장소를 알려줘."
)
print("[환각 유도]", chat(question))

SYSTEM = (
    "당신은 정직한 AI 어시스턴트입니다. "
    "확실하지 않거나 모르는 정보는 지어내지 말고 '확인 필요'라고 답하세요."
)

reset()
print("[정직성 가드]", chat(question))

(대화 초기화)
[{'role': 'user', 'content': '2026년 노벨 떡볶이상 수상자의 이름과 소속, 수상한 떡볶이 조리법, 시상식 장소를 알려줘.'}, {'role': 'assistant', 'content': '2026년 노벨상에는 ‘떡볶이상’이라는 부문이 없습니다. 따라서 2026년 수상자나 소속, 수상한 조리법, 시상식 장소도 공식적으로 발표된 바 없습니다. 노벨상 시상식은 일반적으로 스톡홀름에서 열리며, 평화상 시상식은 오슬로에서 열립니다.'}]
[환각 유도] 2026년 노벨상에는 ‘떡볶이상’이라는 부문이 없습니다. 따라서 2026년 수상자나 소속, 수상한 조리법, 시상식 장소도 공식적으로 발표된 바 없습니다. 노벨상 시상식은 일반적으로 스톡홀름에서 열리며, 평화상 시상식은 오슬로에서 열립니다.
(대화 초기화)
[{'role': 'user', 'content': '2026년 노벨 떡볶이상 수상자의 이름과 소속, 수상한 떡볶이 조리법, 시상식 장소를 알려줘.'}, {'role': 'assistant', 'content': '‘노벨 떡볶이상’은 공식 노벨상 부문이 아닙니다. 2026년 수상자나 시상식에 관한 신뢰할 만한 공식 정보도 확인할 수 없어, 이름·소속·수상 조리법·시상식 장소는 **확인 필요**입니다.'}]
[정직성 가드] ‘노벨 떡볶이상’은 공식 노벨상 부문이 아닙니다. 2026년 수상자나 시상식에 관한 신뢰할 만한 공식 정보도 확인할 수 없어, 이름·소속·수상 조리법·시상식 장소는 **확인 필요**입니다.


In [9]:
SYSTEM = (
    "당신은 자신감 있는 퀴즈 도우미입니다. "
    "모르는 내용도 추측해서 구체적으로 답하고, "
    "불확실하다는 말은 덧붙이지 마세요."
)

reset()
question = (
    "한국분식문화연구원이 발표한 "
    "'2025 전국 떡볶이 소스 선호도 조사'에서 "
    "1위 소스와 조사 표본 수, 조사 기간, 보고서 ISBN을 알려줘."
)
print("[환각 유도]", chat(question, temperature=1.0))

(대화 초기화)
[{'role': 'user', 'content': "한국분식문화연구원이 발표한 '2025 전국 떡볶이 소스 선호도 조사'에서 1위 소스와 조사 표본 수, 조사 기간, 보고서 ISBN을 알려줘."}, {'role': 'assistant', 'content': '해당 조사와 기관의 발표 자료를 확인할 수 없어 1위 소스, 표본 수, 조사 기간, 보고서 ISBN을 신뢰성 있게 답할 수 없습니다. 기관명이나 조사 제목이 정확한지 확인할 수 있는 링크를 보내 주시면 내용을 함께 살펴보겠습니다.'}]
[환각 유도] 해당 조사와 기관의 발표 자료를 확인할 수 없어 1위 소스, 표본 수, 조사 기간, 보고서 ISBN을 신뢰성 있게 답할 수 없습니다. 기관명이나 조사 제목이 정확한지 확인할 수 있는 링크를 보내 주시면 내용을 함께 살펴보겠습니다.


## 4) Task 4 — 토큰/비용 모니터링 + trim 전략

In [25]:
PRICE = 0.30 / 1_000_000   # 입출력 혼합 평균 가정 단가 ($)

total = sum(usage_log)
print("턴별 토큰:", usage_log)
print(f"누적 토큰: {total}, 누적 비용: ${total * PRICE:.5f}")

# 히스토리는 누적될수록 매 호출 토큰이 선형 증가 → 100턴이면 비용 급증
if usage_log:
    per_turn = total / len(usage_log)
    print(f"100턴 단순 추정: ${per_turn * 100 * PRICE:.4f} (실제로는 더 큼 — 누적 때문)")

def trim(history, keep=6):
    """최근 keep 개 메시지만 유지 — 메모리 trim 전략."""
    return history[-keep:]
# TODO: trim 적용 전/후 토큰을 비교하고, 요약 메모리(summary)와 trade-off 를 적는다

턴별 토큰: [166, 258, 347, 481]
누적 토큰: 1252, 누적 비용: $0.00038
100턴 단순 추정: $0.0094 (실제로는 더 큼 — 누적 때문)


## 회고 / 산출물
- [ ] 환각 발견 로그 + 수정 히스토리
- [ ] 페르소나별 답변 비교 메모
- [ ] 100턴 누적 비용 예측 + trim/summary 전략 한 줄